# W13-D6 概念实验：Inventory 三件套的可执行验证——尺子、雷达、排序

配套阅读：`第13周-Day6-VisionCapabilityInventory-技术雷达与演进路线图.md`（md 是盘点与裁决，本 notebook 用**可执行实验**验证三个核心概念）：

1. **实验一 · 立尺子（Sprint 1 理论依据）**：PPV 的 Wilson 置信区间——为什么"误报率 78%"单点数不能交付，样本量决定结论的稳定性（S1 交付判据 = 带置信区间的数字）；
2. **实验二 · 画盘点（交付物代码版）**：Capability Inventory 状态热力图 + 四环 Technology Radar（Trial 空环可视化）；
3. **实验三 · 排序裁决**：三种排序策略（风险/技术/ROI 驱动）的加权评分对比 + 依赖链拓扑约束——验证"前3 Sprint 不含新检测能力"不是拍脑袋。

只用 numpy / matplotlib / 标准库。

In [ ]:
from matplotlib import font_manager
import matplotlib.pyplot as plt
import numpy as np

font_path = "/usr/share/fonts/opentype/noto/NotoSansCJK-Regular.ttc"
font_manager.fontManager.addfont(font_path)
font_name = font_manager.FontProperties(fname=font_path).get_name()
plt.rcParams["font.family"] = font_name
plt.rcParams["axes.unicode_minus"] = False
print("使用字体:", font_name)

## 实验一：Sprint 1 的尺子——PPV 置信区间与样本量

md 第 6 节的最大发现：alert lifecycle 的 `confirmed / false_positive` 标签是运维免费打的监督标签，误报率量化**不需要新采集，只需要统计**。
但统计有个坑：**单点估计会撒谎**。假设系统真实 PPV = 0.78（每 100 个告警 78 个是真的），运维这个月只处理了 40 个告警——从这 40 个里估出的 PPV 可能是 0.70 也可能是 0.86。
Wilson 置信区间比正态近似稳健（小样本/极端比例不越界）。本实验问：**要宣布"误报率 ≤ 22%（即 PPV ≥ 0.78）"，需要多少已处理样本？**

In [ ]:
from math import sqrt

def wilson_ci(k, n, z=1.96):
    """Wilson score interval for k successes in n trials."""
    if n == 0:
        return 0.0, 1.0
    p = k / n
    denom = 1 + z**2 / n
    center = (p + z**2 / (2 * n)) / denom
    half = z * sqrt(p * (1 - p) / n + z**2 / (4 * n**2)) / denom
    return max(0.0, center - half), min(1.0, center + half)

TRUE_PPV = 0.78          # 假设的真实命中率
rng = np.random.default_rng(42)

# 不同样本量下，重复 2000 次抽样，看估计的 PPV 分布有多散
sample_sizes = [20, 50, 100, 300, 1000]
fig, axes = plt.subplots(1, len(sample_sizes), figsize=(16, 3.2), sharey=True)
for ax, n in zip(axes, sample_sizes):
    ests = rng.binomial(n, TRUE_PPV, size=2000) / n
    ax.hist(ests, bins=40, color="#4C78A8", alpha=0.85)
    ax.axvline(TRUE_PPV, color="crimson", lw=1.5, label=f"真实值 {TRUE_PPV}")
    ax.set_title(f"n={n}", fontsize=11)
    ax.set_xlim(0.5, 1.0)
axes[0].set_ylabel("频次"); axes[0].legend(fontsize=8)
fig.suptitle("同一系统（真实 PPV=0.78），样本量决定你嘴里的数字有多可信", fontsize=13)
fig.tight_layout(); fig.savefig("w13d6_ppv_sample_dist.png", dpi=110); plt.close(fig)
print("图已保存: w13d6_ppv_sample_dist.png")

# Wilson 区间宽度 vs 样本量（观测比例就取 0.78）
print(f"{'样本量 n':>8} | {'Wilson 95% CI':>22} | {'区间宽度':>8}")
for n in [20, 50, 100, 300, 1000, 3000]:
    lo, hi = wilson_ci(round(0.78 * n), n)
    print(f"{n:>8} | [{lo:.3f}, {hi:.3f}]".ljust(8) + f" | {hi - lo:>8.3f}")

In [ ]:
# 交付判据的量化：要宣布 "PPV >= 0.78"，观测到的样本比例至少要多少？
# 反过来问：观测 0.82 的命中率，n 多大时 Wilson 下界才 >= 0.78？
target, obs = 0.78, 0.82
need = None
for n in range(10, 5001):
    lo, _ = wilson_ci(round(obs * n), n)
    if lo >= target:
        need = n
        break
print(f"观测 PPV={obs}，要让 Wilson 95% 下界 >= {target}，至少需要 n = {need} 个已判定告警")
print()
print("→ Sprint 1 结论：按 21 摄像头 × 5 规则模板的部署规模，")
print("  每天数十个告警、运维持续打标，1-2 个月即可积累单规则维度的可信样本；")
print("  但『火灾烟雾』这类低频规则可能半年都攒不够 → 这就是 md 里")
print("  火灾漏报率要走『专项抽样评估』而不是等运维标签的原因（频次不够，主动采样补）。")

## 实验二：把两份交付物画出来——Inventory 热力图 + 四环雷达

md 第 6/7 节的两张图在这里用代码复现：
- **热力图**：15 项能力 × 4 档状态（产品化/雏形/无/专项债），按五层+平台分组——一眼看到 L1 满格、L2/L3 空白、L4 雏形、度量栏的 ❌；
- **四环雷达**：Adopt/Trial/Assess/Hold 同心环布局——**Trial 空环是整张图的重点**（技术管道断裂的可视化）。

In [ ]:
# ---- 交付物一：Capability Inventory 状态热力图 ----
# 数据来自 md 第 6 节（每行都过了代码验证）
inv = [
    # (能力, 层, 状态)  状态: 3=产品化 2=雏形 1=规划/无 0=负资产债
    ("障碍物检测 YOLO 11n",      "L1", 3),
    ("零样本检测 YOLO-World",    "L1", 3),
    ("火灾烟雾检测",              "L1", 2),   # 产品化但漏报未量化 → 专项
    ("地面脏污·基线对比",         "L1", 3),
    ("OCR / 分割",               "L1", 1),
    ("目标跟踪 Tracking",        "L2", 1),
    ("场景统计 计数/热区/排队",   "L3", 1),
    ("安全事件运营化",            "L4", 2),
    ("运营 KPI / 指标口径",       "L4", 1),
    ("Vision Agent 推理层",      "L5", 1),
    ("Capability 出口",          "平台", 1),
    ("模型管理/热重载",           "平台", 3),
    ("告警治理 冷却/抑制/生命周期","平台", 3),
    ("通知路由 企微/短信/邮件",   "平台", 3),
    ("误报率度量（尺子）",        "平台", 0),  # 最大 Gap：负资产
]
labels = [f"{r[0]}" for r in inv]
groups = [r[1] for r in inv]
vals   = np.array([r[2] for r in inv], dtype=float)

cmap_vals = {0: 0.0, 1: 0.34, 2: 0.67, 3: 1.0}
m = np.array([[cmap_vals[v] for v in vals]])
fig, ax = plt.subplots(figsize=(9.5, 8))
im = ax.imshow(m, cmap="RdYlGn", aspect=0.35, vmin=0, vmax=1)
ax.set_xticks([0]); ax.set_xticklabels(["状态"], fontsize=10)
ax.set_yticks(range(len(labels)))
ax.set_yticklabels([f"[{g}] {l}" for l, g in zip(labels, groups)], fontsize=10)
status_txt = {0: "债", 1: "— 无", 2: "雏形", 3: "产品化"}
for i, v in enumerate(vals):
    ax.text(0, i, status_txt[int(v)], ha="center", va="center", fontsize=10)
# 分层隔断线
bounds = [i for i in range(1, len(groups)) if groups[i] != groups[i-1]]
for b in bounds:
    ax.axhline(b - 0.5, color="black", lw=1.2)
ax.set_title("Vision Capability Inventory（2026-08 代码实态）\nL1 资产满格 · L2/L3 空白 · L4 雏形 · 平台栏藏着一个『债』（度量）", fontsize=12)
fig.tight_layout(); fig.savefig("w13d6_inventory_heatmap.png", dpi=110); plt.close(fig)
print("图已保存: w13d6_inventory_heatmap.png")
print("分层边界:", [f"{groups[b-1]}|{groups[b]}@{b}" for b in bounds])

In [ ]:
# ---- 交付物二：四环 Technology Radar ----
rings = [("ADOPT 采纳", 1.0, "#2E7D32"), ("TRIAL 试用", 0.75, "#F9A825"),
         ("ASSESS 评估", 0.5, "#EF6C00"), ("HOLD 观望", 0.25, "#B71C1C")]
items = [  # (名称, 环序号0-3, 角度)
    ("YOLO 11n", 0, 30), ("YOLO-World+CLIP", 0, 80), ("基线图像对比", 0, 130), ("CPU-only 部署", 0, 175),
    # Trial 环：空 —— 这是雷达的核心读数
    ("RT-DETR", 2, 30), ("SAM2", 2, 85), ("GroundingDINO", 2, 140),
    ("ByteTrack", 3, 25), ("BoT-SORT", 3, 60), ("ReID", 3, 95), ("Pose", 3, 130), ("VLM", 3, 165),
]
fig, ax = plt.subplots(figsize=(9, 9))
for name, r, color in rings:
    ax.add_patch(plt.Circle((0, 0), r, fill=True, color=color, alpha=0.13, ec=color, lw=1.6))
    ax.text(0, r - 0.06, name, ha="center", fontsize=12, color=color, weight="bold")
for name, ring, deg in items:
    t = np.deg2rad(deg); rr = 0.125 + ring * 0.25
    ax.scatter(rr * np.cos(t), rr * np.sin(t), s=90, color=rings[ring][2])
    ha = "left" if np.cos(t) >= 0 else "right"
    ax.text(rr * np.cos(t) + 0.055 * np.sign(np.cos(t)), rr * np.sin(t), name, ha=ha, va="center", fontsize=10)
# 高亮空 Trial 环
ax.annotate("★ Trial 环全空\n评估→采纳之间的验证管道断了\n（Sprint 1 立尺子 = 装上 Trial 环）",
            xy=(0, -0.62), xytext=(0.42, -0.88), fontsize=10,
            arrowprops=dict(arrowstyle="->", color="#F9A825", lw=2), color="#5D4037")
ax.set_xlim(-1.15, 1.15); ax.set_ylim(-1.05, 1.12); ax.set_aspect("equal"); ax.axis("off")
ax.set_title("MallSenseAI Vision Technology Radar（2026-08）", fontsize=13)
fig.tight_layout(); fig.savefig("w13d6_tech_radar.png", dpi=110); plt.close(fig)
print("图已保存: w13d6_tech_radar.png")
print("环内条目数:", {r[0]: sum(1 for it in items if it[1] == i) for i, r in enumerate(rings)})

## 实验三：排序裁决——三种策略打分 + 依赖链拓扑约束

md 第 8.2 节的裁决在这里变成可计算的：9 个候选 Sprint 项目，每个有（商业价值 V / 风险减免 R / 工程成本 C / 依赖集合 D）。
- **ROI 驱动** ≈ 只看 V/C；
- **技术驱动** ≈ 补五层断层优先（给 L2/L3 加权）；
- **风险驱动（本路线图）** ≈ R 加权 + **依赖链硬约束**（依赖未完成的项目非法）。

看三种策略选出的"前 3"差在哪，以及拓扑约束如何直接判死"先做客流"。

In [ ]:
# 候选项目（来自 md Inventory）：value 0-10, risk_reduction 0-10, cost 1-10（越大越贵）, deps
projects = {
    "S1 度量管线(尺子)":   dict(v=7, r=10, c=3, deps=set()),
    "S1 火灾漏报专项":     dict(v=6, r=9,  c=3, deps=set()),
    "S1 脱敏+LICENSE":     dict(v=4, r=8,  c=2, deps=set()),
    "S2 日级KPI聚合":      dict(v=8, r=5,  c=4, deps={"S1 度量管线(尺子)"}),
    "S2 指标口径层":       dict(v=7, r=6,  c=3, deps={"S1 度量管线(尺子)"}),
    "S3 capability注册表": dict(v=8, r=4,  c=6, deps=set()),
    "S3 alert.query暴露":  dict(v=9, r=5,  c=4, deps={"S3 capability注册表", "S2 指标口径层"}),
    "L2 客流Tracking":     dict(v=10, r=2, c=9, deps={"S1 度量管线(尺子)", "S3 capability注册表"}),
    "VLM 场景理解":        dict(v=8, r=1,  c=8, deps={"L2 客流Tracking", "S3 alert.query暴露"}),
}
def topk(scores, k=3):
    return sorted(scores, key=lambda p: -scores[p])[:k]

# 三种策略的加权评分
strategies = {
    "ROI 驱动 (V/C)":       lambda p: projects[p]["v"] / projects[p]["c"],
    "技术驱动 (断层加权)":   lambda p: (projects[p]["v"] / projects[p]["c"]) * (1.6 if "客流" in p or "VLM" in p else 1.0),
    "风险驱动 (R 优先)":     lambda p: 0.3 * projects[p]["v"] + 0.7 * projects[p]["r"] - 0.4 * projects[p]["c"],
}
fig, axes = plt.subplots(1, 3, figsize=(16, 4.6), sharey=True)
names = list(projects)
colors = {"S1": "#2E7D32", "S2": "#F9A825", "S3": "#1565C0", "L2": "#B71C1C", "VLM": "#6A1B9A"}
for ax, (sname, fn) in zip(axes, strategies.items()):
    sc = {p: fn(p) for p in names}
    order = sorted(names, key=lambda p: sc[p])
    bar_colors = [next((colors[k] for k in ("S1", "S2", "S3", "L2", "VLM") if p.startswith(k)), "grey") for p in order]
    ax.barh(range(len(order)), [sc[p] for p in order], color=bar_colors, alpha=0.85)
    ax.set_yticks(range(len(order))); ax.set_yticklabels(order, fontsize=9)
    top3 = topk(sc)
    for i, p in enumerate(order):
        if p in top3:
            ax.text(sc[p] + 0.15, i, " ←前3", fontsize=9, va="center", weight="bold")
    ax.set_title(sname, fontsize=11)
fig.suptitle("同一张 Inventory，三种排序策略选出完全不同的前3", fontsize=13)
fig.tight_layout(); fig.savefig("w13d6_strategy_compare.png", dpi=110); plt.close(fig)
print("图已保存: w13d6_strategy_compare.png")
for sname, fn in strategies.items():
    sc = {p: round(fn(p), 2) for p in names}
    print(f"\n{sname} 前3:", topk(sc))

In [ ]:
# 依赖链硬约束：拓扑排序下，"先做客流/VLM"直接非法
# Kahn 算法——只有依赖全部完成的项目才可选
def topological_topk(scores, k=3):
    done, picked = set(), []
    cand = set(projects)
    while cand and len(picked) < k:
        ready = [p for p in cand if projects[p]["deps"] <= done]
        if not ready:
            break
        best = max(ready, key=lambda p: scores[p])
        picked.append(best); done.add(best); cand.discard(best)
    return picked

roi_scores = {p: projects[p]["v"] / projects[p]["c"] for p in projects}
print("ROI 驱动·无约束 前3:", topk(roi_scores))
print("ROI 驱动·拓扑约束 前3:", topological_topk(roi_scores))
print()
risk_scores = {p: 0.3 * projects[p]["v"] + 0.7 * projects[p]["r"] - 0.4 * projects[p]["c"] for p in projects}
print("风险驱动·无约束 前3:", topk(risk_scores))
print("风险驱动·拓扑约束 前3:", topological_topk(risk_scores))
print()
print("→ 关键读数：")
print("  1. 拓扑约束一加，客流 Tracking 即使 ROI 最高也只能排后（尺子和注册表是它的依赖）；")
print("  2. 风险驱动选出的前3 恰好 = md 裁决的 S1(尺子+火灾+合规债) → S2(聚合/口径) → S3(接口)，")
print("     且每一步为下一步生产必要条件——这就是『对账→报表→开接口』的可计算形式。")